In [2]:
import numpy as np
import random 
import sys 
import torch
import pandas as pd
from tqdm import tqdm

import analysis_utils as au
from decoderlens import DecoderLens
from extractor import Extractor

sys.path.append("../")

from checkpoint import CheckPoint
from datasets import LetterStringDataLoader
from evaluate import predict_batch

BATCH_SIZE = 1000
%config InlineBackend.figure_format = 'retina'

In [4]:
# load checkpoints:
copy_rand_perm20 = CheckPoint.from_pt("../models/copy_batching_experiments/MLC_batchrand_dallstudy1_copy_perm20_nep20.pt")
rand_perm20 = CheckPoint.from_pt("../models/batching_experiments/MLC_batchrand_dallstudy1_nep20.pt")
alph_perm200 = CheckPoint.from_pt("../models/num_permuted_alphabets/MLC_batchalph_dallstudy1_copy_perm200_nep20.pt")

# load models and dataset:
model_copy = copy_rand_perm20.load_model(verbose=False)
model_unstruct = rand_perm20.load_model(verbose=False)
model_best = alph_perm200.load_model()
dataloader = LetterStringDataLoader(
    mode="test", 
    data_dir="../data/letter-string-analogies/all_transformations_study1_new_alphabets", 
    batch_size=BATCH_SIZE, 
    batching_method="random"
)

Loading model that has completed 19 of 20 epochs
	batch size: 32
	number of steps: 219,659
	best val loss achieved: 0.6038
MLC specs:
	1,400,606 parameters
	3 encoder layers
	3 decoder layers
	8 attention heads
	128 embedding size
	512 feedforward layer sizes
	gelu activation function
	p=0.1 dropout


In [5]:
extractor = Extractor(model_best)
extractor.register()

decoderlens = DecoderLens(model_best)

# set seed for batch
random.seed(10)
batch = next(iter(dataloader))
model_best.eval()
with torch.no_grad():
    out = model_best(batch["yq_io_padded"], batch)

C:\Users\13579681\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\nn\modules\transformer.py:505: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\NestedTensorImpl.cpp:182.)
  output = torch._nested_tensor_from_mask(


In [6]:
for enc_layer in [0,1,2]:
    pred = decoderlens.predict_batch(
        memory=extractor.hidden_states[f"layer_{enc_layer}"],
        batch=batch,
        langs=dataloader.dataset.langs,
        max_length=64,
    )
    correct = [pred == true for pred, true in zip(pred, batch["yq"])]
    print(f"Accuracy using Encoder Layer {enc_layer+1}: {np.mean(correct)}")

Accuracy using Encoder Layer 1: 0.0
Accuracy using Encoder Layer 2: 0.018367346938775512
Accuracy using Encoder Layer 3: 0.6591836734693878


In [30]:
predictions = {
    "transformation": [],
    "problem": [],
    "correct_answer": [],
    "encoder layer 1": [],
    "encoder layer 2": [],
    "encoder layer 3": []
}
num_batches = 1

for i, batch in tqdm(enumerate(dataloader)):
    extractor.clear()
    # append problems and correct answers of current batch
    predictions["transformation"].extend(batch["transformation"])
    predictions["problem"].extend(list(map(lambda x: " ".join(x), batch["xq_context"])))
    predictions["correct_answer"].extend(list(map(lambda x: " ".join(x), batch["yq"])))
    # forward pass to get new hidden states:
    with torch.no_grad():
        out = model_best(batch["yq_io_padded"], batch)
    # take predictions by applying the decoder/out layer to each of the encoder layers:
    for enc_layer in [0,1,2]:
        pred = decoderlens.predict_batch(
            memory=extractor.hidden_states[f"layer_{enc_layer}"],
            batch=batch,
            langs=dataloader.dataset.langs,
            max_length=64,
        )
        predictions[f"encoder layer {enc_layer+1}"].extend(list(map(lambda x: " ".join(x), pred)))

12it [24:54, 124.50s/it]


In [31]:
df_pred = pd.DataFrame(predictions)
df_pred.head()

,transformation,problem,correct_answer,encoder layer 1,encoder layer 2,encoder layer 3
0,extend_group,q t c d e i p h b j k l m y o a f r s n u v w ...,l l l m m m y y y o o o a a a,f,f f m m k l o,l l l m m m y y y o o o
1,succ,a c b v d f g h i j k l m n o p q r s t u e w ...,v d f g h j,f v h i f q,v c f g h i,v d f g h j
2,sort_group,a c b v d f g h i j k l m n o p q r s t u e w ...,p p p q q q r r r s s s t t t,i,q e p e r r s s,p p p q q q r r r s s s t t t
3,fix_pred_succ,a m c d e f g h i j k l b n o p q r s t u v w ...,u p y,u i x f q,t p i v,v w x
4,replicate,x r l b g t e s i j k n m w f p a c h u v d o ...,s i j k n m s i j k n m,f r i f l y i,s i f f k n,s i j k n m w


In [ ]:
df_pred.to_csv("predictions/decoder_lens_new_alphabets.csv", index=False)

In [3]:
# load and compute accuracies:
df_pred = pd.read_csv("predictions/decoder_lens_new_alphabets.csv")
encoder_cols = ['encoder layer 1', 'encoder layer 2', 'encoder layer 3']
accuracies = df_pred[encoder_cols].eq(df_pred['correct_answer'], axis=0).mean().round(3)
accuracies

encoder layer 1    0.000
encoder layer 2    0.019
encoder layer 3    0.665
dtype: float64

In [4]:
df_pred[df_pred["transformation"]=="succ"].head()

,transformation,problem,correct_answer,encoder layer 1,encoder layer 2,encoder layer 3
1,succ,a c b v d f g h i j k l m n o p q r s t u e w ...,v d f g h j,f v h i f q,v c f g h i,v d f g h j
53,succ,a m c d e f g h i j k l b n o p q r s t u v w ...,l b o,f,l m b l,l b o
97,succ,g b c d e f s h i j y l m n o p q r k t u v w ...,e f h,f e s,c f d f,e f h
154,succ,a r c d e f g h i j p l m n o b q y s t u v w ...,d e f h,f o e f q,d e f c,d e f h
161,succ,u q h c p n r v g f k a m w i l j y s t d b o ...,g f k m,t f d k f,g f k f m,g f k m


In [ ]:
n_examples = 20
decoder_lens_pred = decoderlens.predict_batch(
    memory=extractor.hidden_states["layer_2"],
    batch=batch,
    langs=dataloader.dataset.langs,
    max_length=64,
)
for pred, true in zip(decoder_lens_pred[:n_examples], batch["yq"][:n_examples]):
    print(f"pred: {"".join(pred)}, true: {"".join(true)}")

pred: cdefgh, true: cdefgh
pred: sijal, true: sijal
pred: klcnopq, true: klcnopq
pred: qlzls, true: qlzls
pred: cdefyh, true: hyfedc
pred: wzxzyzz, true: wzxzyzz
pred: wqxqyqz, true: wqxqyqz
pred: nooopoqorog, true: nooopoqorog
pred: tvw, true: tvw
pred: lpmpnpo, true: lpmpnpo
pred: gpoxkn, true: gpoxknj
pred: ifr, true: ifr
pred: rygltv, true: hjglta
pred: plmni, true: plmni
pred: stov, true: stov
pred: knmn, true: knmknm
pred: rtunw, true: rtunw
pred: nop, true: nop
pred: ccnnddll, true: ccnnddll
pred: mpztxysn, true: mpztxysnv


In [ ]:
# verify that we get the same predictions for normal model predictions:
extractor.remove()
normal_pred = predict_batch(batch, model_best, dataloader.dataset.langs, max_length=64)
for pred, true in zip(normal_pred[:n_examples], batch["yq"][:n_examples]):
    print(f"pred: {"".join(pred)}, true: {"".join(true)}")

pred: cdefgh, true: cdefgh
pred: sijal, true: sijal
pred: klcnopq, true: klcnopq
pred: qlzls, true: qlzls
pred: cdefyh, true: hyfedc
pred: wzxzyzz, true: wzxzyzz
pred: wqxqyqz, true: wqxqyqz
pred: nooopoqorog, true: nooopoqorog
pred: tvw, true: tvw
pred: lpmpnpo, true: lpmpnpo
pred: gpoxkn, true: gpoxknj
pred: ifr, true: ifr
pred: rygltv, true: hjglta
pred: plmni, true: plmni
pred: stov, true: stov
pred: knmn, true: knmknm
pred: rtunw, true: rtunw
pred: nop, true: nop
pred: ccnnddll, true: ccnnddll
pred: mpztxysn, true: mpztxysnv
